# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mah-gie/Flyrank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Unit of Analysis:** One row represents one unique web page (URL).

**Time Window:** A mid-panel historical month, specifically March 2026 (2026-03), to ensure we don't accidentally train on the final test month.

In [ ]:
from google.colab import userdata
from datasets import load_dataset
import pandas as pd

# Pull your secret token to unlock the data
hf_token = userdata.get('HF_TOKEN')

# Load the warehouse dataset and convert to a pandas dataframe
dataset = load_dataset("FlyRank/internship-warehouse", name="fact_content_daily_performance", split="train", token=hf_token)
df = dataset.to_pandas()

# Filter down to our specific time window
df_march = df[df['month'] == '2026-03'].copy()

print(f"Time window isolated: March 2026.")
print(f"Total rows in this window: {len(df_march)}")

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 4.41MB            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  624kB            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 3.29MB            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 19.6kB            

...

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  146MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/78835655 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

## 2. Fields: feature / label / context / excluded

* **Features:** `content_age_days`, `days_since_last_update`, `impressions_90d`, `avg_position`, `ctr`.

* **Label (Target):** `is_declining` (our binary proxy for decay).

* **Context:** `url` (so we know which page we are looking at).

* **Excluded:** `trend_direction` and `trend_pct`. We exclude these because our label is derived directly from them. Including them as features would cause data leakage.

In [ ]:
features = ['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr']
label = 'is_declining'
context = 'url'
excluded = ['trend_direction', 'trend_pct']

print(f"Model will train on {len(features)} safe features.")

## 3. Verify it with queries (grain, counts, missing values, windows)

Verifying three claims:

1. The grain is truly one row per URL.

2. The row count matches our expectations for a single month.

3. Our feature columns have valid data (no catastrophic missing values).

In [ ]:
# 1. Check the grain (Are URLs completely unique in this month?)
unique_urls = df_march['url'].nunique()
total_rows = len(df_march)
print(f"Grain Check: {total_rows} rows vs {unique_urls} unique URLs. (Should match)")

# 2. Check the label availability
# Creating the proxy label safely
df_march['is_declining'] = df_march['trend_direction'].str.lower().eq('down').astype(int)
print(f"Label Check: {df_march['is_declining'].sum()} pages marked as declining.")

# 3. Check for missing values in our features
print("\nMissing values check:")
print(df_march[features].isna().sum())

## 4. Data limits

**What this data cannot tell us:**

This dataset relies purely on historical Google Search Console metrics. It cannot tell us why a page is declining (e.g., if a competitor published a better article, or if the search intent changed). It also completely misses traffic drops from other sources like social media or direct links. It is purely an indicator of search decay.

In [ ]:
print("Data limits conceptually documented in the markdown cell above.")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.